In [1]:
import pandas as pd
import dask.dataframe as dd
import numpy as np
import os
import datetime as dt
from datetime import date
import gc

In [1]:
year_state_dict_max = {2011: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL',
        'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 
        'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 
        'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 
        'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY'], 
 2012: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL',
        'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 
        'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 
        'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI',
        'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY'], 
 2013: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL',
        'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 
        'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 
        'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 
        'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY'], 
 2014: ['AR', 'AZ', 'CA', 'CT', 'GA', 'HI', 'IA', 'ID', 'IN', 'KY', 
        'LA', 'MA', 'MI', 'MN', 'MO', 'MS', 'NJ', 'NY', 'OH', 'OK',
        'OR', 'PA', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 
        'WV', 'WY'], 
 2015: ['AR', 'CA', 'CT', 'GA', 'IA', 'ID', 'LA', 'MI', 'MN', 'MO',
        'MS', 'NJ', 'NY', 'OR', 'PA', 'SD', 'TN', 'UT', 'VT', 'WV', 'WY']}



In [3]:
def format_benemsis(df, state): 
            
        if df.index.name == 'BENE_MSIS': 
            df = df.reset_index()
        
        else: 
            df['BENE_ID'] = df['BENE_ID'].replace('',np.nan)
            df['BENE_MSIS'] = df['BENE_ID'].fillna(df['MSIS_ID'])
            
        df = df.astype({'BENE_MSIS':'str'})
        
        # if BENE_MSIS begin with state, strip the first five characters from BENE_MSIS


        # if df['BENE_MSIS'].apply(lambda x: x.values[0], meta=('x', df['BENE_MSIS'].dtype)).compute() == f'{state}': 
        if df['BENE_MSIS'].str[:2].iloc[0] == f'{state}':  
            df['BENE_MSIS'] = df['BENE_MSIS'].str[5:]
            
        return df

In [4]:
def format_benemsis_dask(df, state): 
            
        if df.index.name == 'BENE_MSIS': 
            df = df.reset_index()
        
        else: 
            df['BENE_ID'] = df['BENE_ID'].replace('',np.nan)
            df['BENE_MSIS'] = df['BENE_ID'].fillna(df['MSIS_ID'])
            
        df = df.astype({'BENE_MSIS':'str'})
        
        # if BENE_MSIS begin with state, strip the first five characters from BENE_MSIS

        def strip_state_prefix(partition, state):
            if len(partition) == 0:
                return partition
            if partition['BENE_MSIS'].iloc[0].startswith(state):
                partition['BENE_MSIS'] = partition['BENE_MSIS'].str[5:]
            return partition
    
        df = df.map_partitions(strip_state_prefix, state)
                
        return df

In [5]:

def prep_max_files(years, states):

    for year in years: 

        for state in states: 

            print(f'{year}')

            log.write(f'{year}')
            log.write('\n')
            max_lt_cols = ['BENE_ID','MSIS_ID','PRVDR_ID_NMBR','NPI','MSNG_ELG_DATA','MSIS_TOS','MAX_TOS','TYPE_CLM_CD','ADJUST_CD','MDCD_PYMT_AMT','CHRG_AMT','MDCR_COINSUR_PYMT_AMT', 'MDCR_DED_PYMT_AMT', 'TP_PYMT_AMT','PATIENT_LIB_AMT', 'NRSNG_FAC_DAY_CNT','EL_MDCR_XOVR_CLM_BSD_CD','SRVC_BGN_DT','SRVC_END_DT']


            # read in max lt file 

            if (state in ['MD','ME']) & (year==2013): 
                state_lower = state.lower()
                max_lt = pd.read_parquet(f'/gpfs/data/cms-share/data/medicaid/{year}/{state}/max/lt/{state_lower}', columns=max_lt_cols, engine='pyarrow')

            else: 
                max_lt = pd.read_parquet(f'/gpfs/data/cms-share/data/medicaid/{year}/{state}/max/lt/parquet', columns=max_lt_cols, engine='pyarrow')
            
            max_lt = max_lt.reset_index()

            max_lt = format_benemsis(max_lt, state)
            

            print(f'{state} read in')
            log.write(f'{state} read in')
            
            max_lt = pd.DataFrame(max_lt, columns=max_lt.columns)

            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')

            max_lt[['SRVC_BGN_DT', 'SRVC_END_DT']] = max_lt[['SRVC_BGN_DT','SRVC_END_DT']].apply(pd.to_datetime)
            max_lt = max_lt.astype({'MDCD_PYMT_AMT':'float', 'CHRG_AMT':'float', 'MDCR_COINSUR_PYMT_AMT':'float', 'MDCR_DED_PYMT_AMT':'float', 'TP_PYMT_AMT':'float','NRSNG_FAC_DAY_CNT':'int','PATIENT_LIB_AMT':'float'})

            # construct allowed amount 
            max_lt['ALOWD_AMT'] = max_lt['MDCD_PYMT_AMT'] + max_lt['TP_PYMT_AMT'] + max_lt['PATIENT_LIB_AMT']

            
            string_cols = ['BENE_ID','MSIS_ID','PRVDR_ID_NMBR','NPI','MSNG_ELG_DATA','MSIS_TOS','MAX_TOS','TYPE_CLM_CD','ADJUST_CD','EL_MDCR_XOVR_CLM_BSD_CD']
            max_lt[string_cols] = max_lt[string_cols].astype(str)

            # remove claims with missing eligibility data 
            max_lt = max_lt.loc[max_lt['MSNG_ELG_DATA']!=1]
            log.write('removed claims with missing eligibility data')

            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')

            # limit to nursing facility claims 
            max_lt = max_lt.loc[max_lt['MAX_TOS']=='7']
            log.write('limited to nursing facility claims')

            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')

            # remove crossover claims 
            max_lt = max_lt.loc[max_lt['EL_MDCR_XOVR_CLM_BSD_CD']=='0']
            log.write('removed crossover claims')
            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')
            # limit to FFS service claims 
            max_lt = max_lt.loc[max_lt['TYPE_CLM_CD']=='1']
            log.write('limit to FFS claims')
            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')


            # calculate number of days between service begin date and service end date 
            max_lt['DAY_COUNT'] = ((max_lt['SRVC_END_DT'] - max_lt['SRVC_BGN_DT'])/np.timedelta64(1,'D')) + 1

            # remove claims with 0 days
            max_lt = max_lt.loc[max_lt['NRSNG_FAC_DAY_CNT'] > 1]
            log.write('limit to claims with at least 1 day of NRSNG_FAC_DAY_CNT')

            log.write(str(max_lt.shape[0]))
            log.write('\n')

            # drop duplicate claims 

            max_subset = ['BENE_ID','SRVC_BGN_DT','SRVC_END_DT','MDCD_PYMT_AMT','CHRG_AMT']

            log.write('# of rows')
            log.write(str(max_lt.shape[0]))
            log.write('\n')

            max_lt = max_lt.drop_duplicates(subset=max_subset,keep='first')

            log.write('dropped duplicates based on '+str(max_subset))
            log.write('# of rows')
            log.write(str(max_lt.shape[0]))
            log.write('\n')

            print(max_lt.columns)

            # read out file
            max_lt.to_parquet(f'/gpfs/data/cms-share/duas/56930/Nadia/data/medicaid_NF_pmt_rates/max_lt_NF_claims/{year}/{state}')
            log.write('analytical file created')
            log.write('\n')
            log.write('\n')

            log.write('***********************')
            log.write('\n')

    log.close()






In [6]:

def prep_max_files_dask(years, states):
    """ Uses the dask library to prepare the max lt files for rate calculation and writes script progress to a log file, which must be opened before the function is run."""

    for year in years: 

        for state in states: 

            print(f'{year}')

            log.write(f'{year}')
            log.write('\n')
            max_lt_cols = ['BENE_ID','MSIS_ID','PRVDR_ID_NMBR','NPI','MSNG_ELG_DATA','MSIS_TOS','MAX_TOS','TYPE_CLM_CD','ADJUST_CD','MDCD_PYMT_AMT','CHRG_AMT','MDCR_COINSUR_PYMT_AMT', 'MDCR_DED_PYMT_AMT', 'TP_PYMT_AMT','PATIENT_LIB_AMT', 'NRSNG_FAC_DAY_CNT','EL_MDCR_XOVR_CLM_BSD_CD','SRVC_BGN_DT','SRVC_END_DT']


            # read in max lt file
            
            max_lt = dd.read_parquet(f'{directory_for_max_lt_data_in_parquet_format}/{state}/max/lt/parquet', columns=max_lt_cols, engine='pyarrow')
            
            max_lt = max_lt.reset_index()

            max_lt = format_benemsis_dask(max_lt, state)
            

            print(f'{state} read in')
            log.write(f'{state} read in')
            
            max_lt = pd.DataFrame(max_lt, columns=max_lt.columns)

            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')

            max_lt[['SRVC_BGN_DT', 'SRVC_END_DT']] = max_lt[['SRVC_BGN_DT','SRVC_END_DT']].apply(pd.to_datetime)
            max_lt = max_lt.astype({'MDCD_PYMT_AMT':'float', 'CHRG_AMT':'float', 'MDCR_COINSUR_PYMT_AMT':'float', 'MDCR_DED_PYMT_AMT':'float', 'TP_PYMT_AMT':'float','NRSNG_FAC_DAY_CNT':'int','PATIENT_LIB_AMT':'float'})

            # construct allowed amount 
            max_lt['ALOWD_AMT'] = max_lt['MDCD_PYMT_AMT'] + max_lt['TP_PYMT_AMT'] + max_lt['PATIENT_LIB_AMT']

            
            string_cols = ['BENE_ID','MSIS_ID','PRVDR_ID_NMBR','NPI','MSNG_ELG_DATA','MSIS_TOS','MAX_TOS','TYPE_CLM_CD','ADJUST_CD','EL_MDCR_XOVR_CLM_BSD_CD']
            max_lt[string_cols] = max_lt[string_cols].astype(str)

            # remove claims with missing eligibility data 
            max_lt = max_lt.loc[max_lt['MSNG_ELG_DATA']!=1]
            log.write('removed claims with missing eligibility data')

            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')

            # limit to nursing facility claims 
            max_lt = max_lt.loc[max_lt['MAX_TOS']=='7']
            log.write('limited to nursing facility claims')

            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')

            # remove crossover claims 
            max_lt = max_lt.loc[max_lt['EL_MDCR_XOVR_CLM_BSD_CD']=='0']
            log.write('removed crossover claims')
            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')
            # limit to FFS service claims 
            max_lt = max_lt.loc[max_lt['TYPE_CLM_CD']=='1']
            log.write('limit to FFS claims')
            log.write(str(max_lt.shape[0]) + ('claims'))
            log.write('\n')


            # calculate number of days between service begin date and service end date 
            max_lt['DAY_COUNT'] = ((max_lt['SRVC_END_DT'] - max_lt['SRVC_BGN_DT'])/np.timedelta64(1,'D')) + 1

            # remove claims with 0 days
            max_lt = max_lt.loc[max_lt['NRSNG_FAC_DAY_CNT'] > 1]
            log.write('limit to claims with at least 1 day of NRSNG_FAC_DAY_CNT')

            log.write(str(max_lt.shape[0]))
            log.write('\n')

            # drop duplicate claims 

            max_subset = ['BENE_ID','SRVC_BGN_DT','SRVC_END_DT','MDCD_PYMT_AMT','CHRG_AMT']

            log.write('# of rows')
            log.write(str(max_lt.shape[0]))
            log.write('\n')

            max_lt = max_lt.drop_duplicates(subset=max_subset,keep='first')

            log.write('dropped duplicates based on '+str(max_subset))
            log.write('# of rows')
            log.write(str(max_lt.shape[0]))
            log.write('\n')

            print(max_lt.columns)

            # read out file
            max_lt.to_parquet(f'/{directory_for_processed_datasets}/max_lt_NF_claims/{year}/{state}')
            log.write('analytical file created')
            log.write('\n')
            log.write('\n')

            log.write('***********************')
            log.write('\n')

    log.close()




SyntaxError: invalid syntax. Perhaps you forgot a comma? (2532491191.py, line 25)

In [16]:
#### Example: Run this function a subset of 2015 MAX state files. 

years = [2015]

log = open(f'/{directory_for_log_files}/01_prep_max_files_2011-2015_{date.today()}.txt', "w")

states_2015 = ['AR', 'CA', 'CT', 'GA', 'IA', 'ID', 'LA', 'MI', 'MN', 'MO','MS', 'NJ', 'OR', 'PA', 'SD', 'TN', 'UT', 'VT', 'WV', 'WY']
prep_max_files(years, states_2015)

log.close()


2015
AR read in
Index(['BENE_MSIS', 'BENE_ID', 'MSIS_ID', 'PRVDR_ID_NMBR', 'NPI',
       'MSNG_ELG_DATA', 'MSIS_TOS', 'MAX_TOS', 'TYPE_CLM_CD', 'ADJUST_CD',
       'MDCD_PYMT_AMT', 'CHRG_AMT', 'MDCR_COINSUR_PYMT_AMT',
       'MDCR_DED_PYMT_AMT', 'TP_PYMT_AMT', 'PATIENT_LIB_AMT',
       'NRSNG_FAC_DAY_CNT', 'EL_MDCR_XOVR_CLM_BSD_CD', 'SRVC_BGN_DT',
       'SRVC_END_DT', 'ALOWD_AMT', 'DAY_COUNT'],
      dtype='object')
2015


KeyboardInterrupt: 